In [0]:
dbutils.widgets.removeAll()

In [0]:
# Create widget for ProcessedJSON parameter
dbutils.widgets.text("ProcessedJSON", "", "")
widget_value = dbutils.widgets.get("ProcessedJSON")

if widget_value:
    ProcessedJSON = widget_value

In [0]:
import json
from pyspark.sql.functions import explode, col

In [0]:
%run "./MoveFileToProcess"

In [0]:
%run "../CommonMethods/Helpers/SynJSONCreatorClass"

In [0]:
%run "../CommonMethods/Helpers/FileHandling"

In [0]:
if isinstance(ProcessedJSON, str):
    if not ProcessedJSON or ProcessedJSON.strip() == "":
        print("No ProcessedJSON parameter provided. Exiting.")
        dbutils.notebook.exit(json.dumps([]))
    ProcessedJSON = json.loads(ProcessedJSON)
elif not ProcessedJSON:
    print("No ProcessedJSON parameter provided. Exiting.")
    dbutils.notebook.exit(json.dumps([]))

filesDF = spark.createDataFrame([ProcessedJSON])
filesDF.show(truncate=False)
explodedFileIDs = filesDF.select(explode(col("FileIds"))).select(
    col("col.ClientID"),
    col("col.FileID"),
    col("col.FileName"),
    col("col.ClientContainer"),
    col("col.CurrentFolderPath"),
    col("col.ProcessedFolderPath"),
    col("col.ColumnDelimiter"),
    col("col.HasHeader"),
    col("col.IgnoreHeader"),
    col("col.FileLayoutID"),
    col("col.FileLayoutDescription"),
    col("col.SchemaFileName"),
    col("col.SchemaFilePath"),
    col("col.TextQualifier")
)

ErrorMessage = ""
doubleQuote = '"'

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
try:
    currentJobId = ctx.tags().get("jobId").getOrElse(lambda: "Undefined")
except Exception:
    currentJobId = "Undefined"

rJSON = synJSONCreator()
rJSON.addBracketStart()

if explodedFileIDs.count() == 0:
    raise ValueError("No records found in explodedFileIDs. Please check the upstream payload execution.")

# Loop through all files to process each one
for row in explodedFileIDs.collect():
    print(f"Begin Processing: {row['FileID']}-{row['FileName']}")
    
    CurrPath = f"{row['ClientContainer']}{row['CurrentFolderPath']}"
    ProcessedPath = row['ProcessedFolderPath']
    SchemaFile = f"{row['SchemaFilePath']}/{row['SchemaFileName']}"
    FullFileName = f"{row['ClientContainer']}{row['CurrentFolderPath']}/{row['FileName']}"
    
    print(f"--> Target Data Path: file:{FullFileName}")
    print(f"--> Target Schema Path: file:{SchemaFile}")
    print(f"--> Target Output Path: {ProcessedPath}")
    
    # Validate Data File Presence
    try:
        # Handle workspace paths and Volume paths correctly
        if FullFileName.startswith('/Volumes/') or FullFileName.startswith('/Workspace/'):
            check_path = FullFileName
        else:
            check_path = f"file:{FullFileName}"
        dbutils.fs.ls(check_path)
        is_data_file_valid = True
    except Exception as e:
        print(f"Data file check failed: {str(e)}")
        is_data_file_valid = False
    
    # Validate Schema File Presence
    try:
        # Handle workspace paths correctly
        if SchemaFile.startswith('/Volumes/') or SchemaFile.startswith('/Workspace/'):
            check_schema = SchemaFile
        else:
            check_schema = f"file:{SchemaFile}"
        dbutils.fs.ls(check_schema)
        is_schema_file_valid = True
    except Exception as e:
        print(f"Schema file check failed: {str(e)}")
        is_schema_file_valid = False
    
    # Execute Single Mapping Event
    if is_data_file_valid and is_schema_file_valid:
        rJSON.addBraceStart()
        rJSON.addNewEntry("FileID", row['FileID'])
        rJSON.addNewEntry("FileName", row['FileName'])
        
        try:
            results = process_move_file(
                ClientId=row['ClientID'],
                FileId=row['FileID'],
                FileLayoutId=row['FileLayoutID'],
                FileLayoutDescription=row['FileLayoutDescription'],
                ColumnDelimiter=row['ColumnDelimiter'],
                HasHeader=row['HasHeader'],
                IgnoreHeader=row['IgnoreHeader'],
                textQualifier=row['TextQualifier'],
                FullFileName=FullFileName,
                SchemaFile=SchemaFile,
                ProcessedPath=ProcessedPath
            )
    
            returnedJson = spark.createDataFrame([json.loads(str(results))])
    
            for x in returnedJson.collect():
                rJSON.addNewEntry("FullFilePath", ProcessedPath)
                rJSON.addNewEntry("CurrentJobId", x['CurrentJobId'])
                rJSON.addNewEntry("Status", x['Status']) 
                rJSON.addNewEntry("RecordCount", x['ProcessedCount'])     
                rJSON.addNewEntry("ErrorMessage", x['ErrorMessage'], False)                                    
                
        except Exception as e:
            clean_err = str(e).strip().replace(doubleQuote, "")
            rJSON.addNewEntry("CurrentJobId", "Undefined")
            rJSON.addNewEntry("Status", "FAILURE")
            rJSON.addNewEntry("RecordCount", "")   
            rJSON.addNewEntry("ErrorMessage", clean_err, False)   
    
        rJSON.addBraceEnd()
    
    elif not is_data_file_valid:
        rJSON.addBraceStart()
        rJSON.addNewEntry("CurrentJobId", "Undefined")
        rJSON.addNewEntry("FileID", row['FileID'])
        rJSON.addNewEntry("FileName", row['FileName'])
        rJSON.addNewEntry("FullFilePath", CurrPath)
        rJSON.addNewEntry("Status", "FAILED")
        rJSON.addNewEntry("RecordCount", "")   
        rJSON.addNewEntry("ErrorMessage", "Data File Not Found", False)
        rJSON.addBraceEnd()
        
    elif not is_schema_file_valid:
        rJSON.addBraceStart()
        rJSON.addNewEntry("CurrentJobId", "Undefined")
        rJSON.addNewEntry("FileID", row['FileID'])
        rJSON.addNewEntry("FileName", row['SchemaFileName'])
        rJSON.addNewEntry("FullFilePath", CurrPath)
        rJSON.addNewEntry("Status", "FAILED")
        rJSON.addNewEntry("RecordCount", "")   
        rJSON.addNewEntry("ErrorMessage", "Schema File Not Found", False)
        rJSON.addBraceEnd()
    
    # Add comma separator between file results (except after the last one)
    if row != explodedFileIDs.collect()[-1]:
        rJSON.addComma()

rJSON.addBracketEnd()

returnVal = rJSON.getJSON()
print(returnVal)

dbutils.notebook.exit(returnVal)

In [0]:
import json

# Input CSV path and Target Volume path
csv_file_path = "/Workspace/Repos/logi@openhealthagents.org/claimspan/ClaimsProcessing/temp/834/angelina.csv"
processed_volume_path = "/Volumes/claimspan/bronze/member"
schema_dir = "/Workspace/Repos/logi@openhealthagents.org/claimspan/ClaimsProcessing/DimMember/Bronze/Schema"

# Build payload
payload = json.dumps({
    "FileIds": [{
        "ClientID": "TEST_CLIENT",
        "FileID": "1001",
        "FileName": "angelina.csv",
        "ClientContainer": "/Workspace/Repos/logi@openhealthagents.org/claimspan/ClaimsProcessing/temp/834",
        "CurrentFolderPath": "",
        "ProcessedFolderPath": processed_volume_path,
        "ColumnDelimiter": ",",
        "HasHeader": "true",
        "IgnoreHeader": "False",
        "FileLayoutID": "834",
        "FileLayoutDescription": "Standard834",
        "SchemaFileName": "MemberSchema.json",
        "SchemaFilePath": schema_dir,
        "TextQualifier": "\""
    }]
})

if __name__ == "__main__":
    notebook_path = "/Workspace/Repos/logi@openhealthagents.org/claimspan/ClaimsProcessing/Shared/Notebooks/FilesToProcess"
    
    result = dbutils.notebook.run(notebook_path, 600, {"ProcessedJSON": payload})
    print("Result:", result)